# Silver to Gold: Modelagem Dimensional (Star Schema) + Contexto GenAI

Este notebook consolida os dados da camada Silver em um modelo dimensional Star Schema otimizado para BI, além de gerar a tabela de contexto `gold_genai_movies_context` para o Vector Search do assistente de IA.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

catalog = "cinedata_analytics"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

DataFrame[]

## 1. Dimensões (Star Schema)

As dimensões armazenam os metadados descritivos de cada entidade de negócio. As chaves substitutas (Surrogate Keys) são geradas via `row_number()` para acelerar os JOINs analíticos.

In [0]:
w_movie = Window.orderBy("id_filme")

dim_movies = (
    spark.table(f"{silver_schema}.tb_info_filmes")
    .withColumn("sk_movie_id", F.row_number().over(w_movie).cast("bigint"))
    .select(
        "sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
        "duracao_minutos", "idioma_original", "status_filme", "sinopse"
    )
)

dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_movies")
print(f"✅ dim_movies gravada: {dim_movies.count()} registros")

dim_movies_lk = dim_movies.select("sk_movie_id", "id_filme")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ dim_movies gravada: 93220 registros


In [0]:
w_genre = Window.orderBy("nome_genero")

dim_genres = (
    spark.table(f"{silver_schema}.tb_generos")
    .select("nome_genero")
    .distinct()
    .withColumn("sk_genre_id", F.row_number().over(w_genre).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_genres")
print(f"✅ dim_genres gravada: {dim_genres.count()} registros")

w_person = Window.orderBy("nome_entidade", "tipo_entidade")

dim_people = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .select("nome_entidade", "tipo_entidade")
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(w_person).cast("bigint"))
    .select(
        "sk_person_id",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
)

dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_people")
print(f"✅ dim_people gravada: {dim_people.count()} registros")

w_company = Window.orderBy("nome_entidade")

dim_companies = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("nome_entidade")
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(w_company).cast("bigint"))
    .select("sk_company_id", F.col("nome_entidade").alias("nome_produtora"))
)

dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_companies")
print(f"✅ dim_companies gravada: {dim_companies.count()} registros")

✅ dim_genres gravada: 19 registros
✅ dim_people gravada: 418489 registros
✅ dim_companies gravada: 45546 registros


## 2. Tabela Fato e Dimensão de Reviews

- **fact_movies_performance**: Grão de um registro por filme. Centraliza métricas financeiras e de engajamento.
- **dim_reviews**: Consolida as avalições individuais em uma métrica resumida por filme.

In [0]:
df_fact = (
    dim_movies_lk
    .join(spark.table(f"{silver_schema}.tb_financeiro_filmes"), "id_filme", "left")
    .join(spark.table(f"{silver_schema}.tb_metricas_engajamento"), "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
)

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_movies_performance")
print(f"✅ fact_movies_performance gravada: {df_fact.count()} registros")

reviews_agg = (
    spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )
)

w_review = Window.orderBy("id_filme")
dim_reviews = (
    dim_movies_lk
    .join(reviews_agg, "id_filme", "left")
    .withColumn("sk_review_id", F.row_number().over(w_review).cast("bigint"))
    .select(
        "sk_review_id", "sk_movie_id",
        F.coalesce(F.col("qtd_avaliacoes_usuarios"), F.lit(0)).cast("int").alias("qtd_avaliacoes_usuarios"),
        F.col("nota_media_usuarios").cast("double")
    )
)

dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_reviews")
print(f"✅ dim_reviews gravada: {dim_reviews.count()} registros")

✅ fact_movies_performance gravada: 93220 registros
✅ dim_reviews gravada: 93220 registros


## 3. Tabelas Ponte (Bridge Tables)

Conectam a tabela `dim_movies` às dimensões periféricas (`dim_genres`, `dim_people`, `dim_companies`) sem duplicar o grão da tabela Fato.

In [0]:
bridge_movie_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .join(dim_movies, "id_filme", "inner")
    .join(dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)
bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_genre")
print(f"✅ bridge_movie_genre gravada: {bridge_movie_genre.count()} registros")

bridge_movie_person = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .join(dim_movies, "id_filme", "inner")
    .join(dim_people,
          (F.col("nome_entidade") == F.col("nome_pessoa")) &
          (F.col("tipo_entidade") == F.col("tipo_pessoa")), "inner")
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)
bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_person")
print(f"✅ bridge_movie_person gravada: {bridge_movie_person.count()} registros")

bridge_movie_company = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(dim_movies, "id_filme", "inner")
    .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"), "inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)
bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_company")
print(f"✅ bridge_movie_company gravada: {bridge_movie_company.count()} registros")

✅ bridge_movie_genre gravada: 133737 registros
✅ bridge_movie_person gravada: 721683 registros
✅ bridge_movie_company gravada: 113407 registros


## 4. Tabela de Contexto GenAI (gold_genai_movies_context)

Gera um documento textual consolidado por filme para alimentar o Vector Search do assistente de IA (RAG).

**Tratamento de Nulos**: Funções `concat()` e `||` retornam NULL se qualquer campo for nulo. Por isso, cada campo é envolvido por `F.coalesce()` com um texto de fallback apropriado, garantindo que nenhum filme desapareca da tabela de contexto.

In [0]:
print("\nProcessando: gold_genai_movies_context...")

actors_per_movie = (
    bridge_movie_person
    .join(dim_people, "sk_person_id", "inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.collect_list("nome_pessoa").alias("atores_arr"))
    .withColumn("atores_str", F.concat_ws(", ", F.slice(F.col("atores_arr"), 1, 5)))
    .select("sk_movie_id", "atores_str")
)

directors_per_movie = (
    bridge_movie_person
    .join(dim_people, "sk_person_id", "inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.collect_list("nome_pessoa").alias("diretores_arr"))
    .withColumn("diretores_str", F.concat_ws(", ", F.col("diretores_arr")))
    .select("sk_movie_id", "diretores_str")
)

gold_genai_movies_context = (
    dim_movies
    .join(spark.table(f"{gold_schema}.fact_movies_performance"), "sk_movie_id", "left")
    .join(actors_per_movie, "sk_movie_id", "left")
    .join(directors_per_movie, "sk_movie_id", "left")
    .withColumn("llm_context_document",
        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("titulo"), F.lit("título não disponível")),
            F.lit(", lançado no ano de "),
            F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano desconhecido")),
            F.lit(", faturou "),
            F.coalesce(F.col("receita_brl").cast("string"), F.lit("receita não informada")),
            F.lit(" e teve um custo de "),
            F.coalesce(F.col("orcamento_brl").cast("string"), F.lit("orçamento não informado")),
            F.lit(". Estrelado por "),
            F.coalesce(F.col("atores_str"), F.lit("elenco não disponível")),
            F.lit(" e dirigido por "),
            F.coalesce(F.col("diretores_str"), F.lit("diretor não disponível")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.col("sinopse"), F.lit("sinopse não disponível")),
            F.lit(".")
        )
    )
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        "llm_context_document"
    )
)

gold_genai_movies_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.gold_genai_movies_context")
print(f"✅ gold_genai_movies_context gravada: {gold_genai_movies_context.count()} registros")
display(gold_genai_movies_context.limit(5))


Processando: gold_genai_movies_context...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ gold_genai_movies_context gravada: 93220 registros


movie_id,title,llm_context_document
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por João Pedro Bénard, Isabel Abreu, Marcello Urgeghe, Inês Pronto e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Claire Chust, Maxime Pambet, Gabrielle Cohen, Mouss Zouheyri, Soulayman Rkiba e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: sinopse não disponível."
1000030,58 Hours: The Baby Jessica Story,"O filme 58 Hours: The Baby Jessica Story, lançado no ano de 2021, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por elenco não disponível e dirigido por Mark Bone, o filme possui a seguinte sinopse: sinopse não disponível."


## 5. Desafio de Analytics

Validação do modelo Gold através de perguntas de negócio recorrentes na rotina de um Analista de Dados. Resultados exibidos via `display()`.

In [0]:
print("\n### a) Receita Total (BRL)")
display(spark.sql(f"""
    SELECT SUM(receita_brl) AS receita_total_brl
    FROM {gold_schema}.fact_movies_performance
"""))


### a) Receita Total (BRL)


receita_total_brl
808261384410.12


In [0]:
print("\n### b) Top 5 Filmes por Popularidade")
display(spark.sql(f"""
    SELECT m.titulo, f.popularidade
    FROM {gold_schema}.fact_movies_performance f
    JOIN {gold_schema}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))


### b) Top 5 Filmes por Popularidade


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


In [0]:
print("\n### c) Filmes por Gênero")
display(spark.sql(f"""
    SELECT g.nome_genero, COUNT(*) AS qtd_filmes
    FROM {gold_schema}.bridge_movie_genre b
    JOIN {gold_schema}.dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))


### c) Filmes por Gênero


nome_genero,qtd_filmes
Drama,30729
Documentary,17987
Comedy,17721
Thriller,9794
Horror,9225
Romance,7298
Action,5775
Crime,4525
Animation,4280
TV Movie,3909


In [0]:
print("\n### d) Top 10 Filmes por Receita (com RANK)")
display(spark.sql(f"""
    SELECT m.titulo, f.receita_usd, f.receita_brl,
           RANK() OVER (ORDER BY f.receita_usd DESC) AS ranking
    FROM {gold_schema}.fact_movies_performance f
    JOIN {gold_schema}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    ORDER BY f.receita_usd DESC
    LIMIT 10
"""))


### d) Top 10 Filmes por Receita (com RANK)


titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,14311080000.00,1
Avatar: The Way of Water,2320250281.00,11859031211.22,2
AVENGERS: INFINITY WAR,2052415039.00,10490098505.83,3
spider-man: no way home,1921847111.00,9822752769.03,4
The Lion King,1663075401.00,8500144682.05,5
Top Gun: Maverick,1488732821.00,7609062321.41,6
Barbie,1428545028.00,7301436492.61,7
The Super Mario Bros. Movie,1355725263.00,6929247391.72,8
Black Panther,1349926083.00,6899607202.82,9
Star Wars: The Last Jedi,1332698830.00,6811556990.01,10


In [0]:
max_release = spark.sql(f"""
    SELECT MAX(data_lancamento) AS max_date
    FROM {gold_schema}.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= CURRENT_DATE()
""").collect()[0][0]

print(f"\n### e) Ator com mais participações (últimos 2 anos a partir de {max_release})")
display(spark.sql(f"""
    WITH filmes_recentes AS (
        SELECT sk_movie_id
        FROM {gold_schema}.dim_movies
        WHERE status_filme = 'Lançado'
          AND data_lancamento >= ADD_MONTHS('{max_release}', -24)
    )
    SELECT p.nome_pessoa AS ator, COUNT(*) AS qtd_participacoes
    FROM {gold_schema}.bridge_movie_person b
    JOIN {gold_schema}.dim_people p ON b.sk_person_id = p.sk_person_id
    JOIN filmes_recentes f ON b.sk_movie_id = f.sk_movie_id
    WHERE p.tipo_pessoa = 'Ator'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC
    LIMIT 1
"""))


### e) Ator com mais participações (últimos 2 anos a partir de 2026-02-19)


ator,qtd_participacoes
Tristan Welsh,11


In [0]:
print(f"\n### f) Produtora com maior Lucro (últimos 5 anos a partir de {max_release})")
display(spark.sql(f"""
    WITH filmes_recentes AS (
        SELECT sk_movie_id
        FROM {gold_schema}.dim_movies
        WHERE status_filme = 'Lançado'
          AND data_lancamento >= ADD_MONTHS('{max_release}', -60)
    )
    SELECT c.nome_produtora, SUM(f.lucro_brl) AS lucro_total_brl
    FROM {gold_schema}.bridge_movie_company b
    JOIN {gold_schema}.dim_companies c ON b.sk_company_id = c.sk_company_id
    JOIN filmes_recentes fr ON b.sk_movie_id = fr.sk_movie_id
    JOIN {gold_schema}.fact_movies_performance f ON b.sk_movie_id = f.sk_movie_id
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_brl DESC
    LIMIT 1
"""))


### f) Produtora com maior Lucro (últimos 5 anos a partir de 2026-02-19)


nome_produtora,lucro_total_brl
Universal Pictures,29502954222.35
